# Wealth Transfer Model (Mesa 2.x)

This notebook implements the classic Boltzmann wealth-transfer model with Mesa's
agent-based modeling framework. Every agent starts with one unit of wealth and,
on each step, an agent with wealth gives one unit to another randomly chosen
agent. Over many steps the wealth distribution develops a skewed, inequality-like
shape. It is a compact, self-contained teaching example written against the
Mesa 2.x API.

In [ ]:
# Install the exact Mesa version this notebook was written against.
#!pip install mesa==2.1.5

In [ ]:
import mesa
from mesa.time import RandomActivation

## The scheduler

The model uses the RandomActivation scheduler which activates agents in random
order each step. Shuffling the activation order avoids the systematic bias that a
fixed order would introduce.

In [ ]:
class MoneyAgent(mesa.Agent):
    """An agent with a single unit of wealth to give away."""

    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)
        self.wealth = 1

    def step(self):
        if self.wealth > 0:
            other = self.random.choice(self.model.schedule.agents)
            other.wealth += 1
            self.wealth -= 1

In [ ]:
class MoneyModel(mesa.Model):
    """A model with N wealth-holding agents on a random-activation schedule."""

    def __init__(self, N):
        super().__init__()
        self.num_agents = N
        self.schedule = RandomActivation(self)
        for i in range(self.num_agents):
            a = MoneyAgent(i, self)
            self.schedule.add(a)
        self.datacollector = mesa.DataCollector(agent_reporters={"Wealth": "wealth"})

    def step(self):
        self.datacollector.collect(self)
        self.schedule.step()

In [ ]:
model = MoneyModel(50)
for _ in range(20):
    model.step()

print("The scheduler advanced", model.schedule.steps, "steps.")

In [ ]:
results = mesa.batch_run(
    MoneyModel,
    parameters={"N": range(10, 50, 10)},
    iterations=5,
    max_steps=20,
)
print(len(results), "rows of results collected.")

## Notes

The model above is written entirely against the Mesa 2.x API: agents receive an
explicit `unique_id`, activation is delegated to a `mesa.time` scheduler, and
`batch_run` sweeps the population size. These are exactly the conventions a later
Mesa release will need to modernise.